In [1]:
import scanpy as sc
import hdf5plugin
import anndata as ad

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn.objects  as so

from scipy.stats import chi2_contingency
import decoupler as dc

from tqdm import tqdm

In [163]:
import importlib
importlib.reload(dc)

<module 'decoupler' from '/storage/mi/sommereg03/vbsc_2/lib/python3.13/site-packages/decoupler/__init__.py'>

# Load Data:

In [2]:
def ctrl_pert_split_dataset(norm_data: ad.AnnData) -> tuple[list[str], dict[str,list[str]]]:
    # Split df into perturbed and control cells
    by_pert = [pert for _, pert in norm_data.obs.groupby(norm_data.obs["perturbation"])]
    # Find indices for each perturbation
    pert_i_dict = {pert.iloc[0]["perturbation"]:pert.index.to_list() for pert in by_pert}
    # Extract the control cells separately
    ctrl_i_vec = pert_i_dict.pop("ctrl")
    return (ctrl_i_vec, pert_i_dict)

In [4]:
names = ["Norman19"]#,"Replogle22"]
models = ["experimental"]#,"pGRiNS"]#, "Random"]

In [5]:
data = {}
for name in names:
    data[name] = {}
    for model in models:
        data[name][model] = {}
    data[name]["experimental"]["adata"] = sc.read_h5ad(f"../Data/Experimental/{name}/perturb_norm_subset_KeggoRo.h5ad")
    perts = list(data[name]["experimental"]["adata"].obs["perturbation"].unique())[:10]
    data[name]["experimental"]["adata"] = data[name]["experimental"]["adata"][data[name]["experimental"]["adata"].obs["perturbation"].isin(["ctrl"]+perts)]
"""
pgrins_full = sc.read_h5ad("../Data/Projects/KeggoRo/perturb_norm_pert_reduced.h5ad")
data["Norman19"]["pGRiNS"]["adata"] = pgrins_full[pgrins_full.obs["PertNum"]<=75]
data["Replogle22"]["pGRiNS"]["adata"] = pgrins_full[pgrins_full.obs["PertNum"]==-1 | pgrins_full.obs["PertNum"]>75]
"""

for name in names:
    data[name]["perts"] = list(data[name]["experimental"]["adata"].obs["perturbation"].unique())
    data[name]["perts"].remove("ctrl")
    for model in models:
        data[name][model]["pert_indices"] = ctrl_pert_split_dataset(data[name][model]["adata"])
        data[name][model]["pert_means"] = {pert : np.asarray(np.mean(data[name][model]["adata"][data[name][model]["pert_indices"][1][pert]].layers["log1p"],axis=0)).squeeze() for pert in tqdm(data[name]["perts"])}

/tmp/ipykernel_79572/3870400248.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_pert = [pert for _, pert in norm_data.obs.groupby(norm_data.obs["perturbation"])]
100%|██████████| 10/10 [00:06<00:00,  1.46it/s]


# Visual comparison:

In [ ]:
# visualization method: heatmap

# Shared DEGs:

In [16]:
for name in names:
    for model in models:
        sc.tl.rank_genes_groups(data[name][model]["adata"],groupby="perturbation",reference="ctrl",key_added="DEG_analysis_ctrl",rankby_abs=True,layer="log1p")
        data[name][model]["DEGs"] = {pert : data[name][model]["adata"].uns["DEG_analysis_ctrl"]["names"][pert][data[name][model]["adata"].uns["DEG_analysis_ctrl"]["pvals_adj"][pert] < 0.05] for pert in tqdm(data[name]["perts"])}
        data[name][model]["ON_genes"] = {pert : data[name][model]["adata"].var_names[data[name][model]["pert_means"][pert]>np.mean(data[name][model]["pert_means"][pert])] for pert in tqdm(data[name]["perts"])}

100%|██████████| 10/10 [00:00<00:00, 3793.69it/s]


In [ ]:
chisq = {}
for test in ["DEGs","ON_genes"]:
    chisq[test] = {}
    for name in names:
        chisq_res[test][name] = []
        for pert in data[name]["perts"]:
            genes_exp_set = set(data[name]["experimental"][test])
            genes_pgrins_set = set(data[name]["pGRiNS"][test])
            all_genes = set(data[name]["experimental"]["adata"].var_names)

            # Get contingency matrix
            count_matrix = np.array([[len(genes_exp_set & genes_pgrins_set),len(genes_pgrins_set - genes_exp_set)],[len(genes_exp_set - genes_pgrins_set),len(all_genes - (genes_exp_set|genes_pgrins_set))]])
            res = chi2_contingency(count_matrix)
            chisq_res[test][name].append(res.pvalue)


In [ ]:
# plot chi squared pval hist for perts
# for comparison: randomly assign genes as DEGs and calculate chi squared stat between that and exp data -> maybe unnecessary since were plotting pvals anyways?
# then do t test (or wilcoxon or sth) between distributions

# Compare using metrics:

In [41]:
for name in names:
    sc.tl.rank_genes_groups(data[name]["experimental"]["adata"],groupby="perturbation",reference="rest",key_added="DEG_analysis_Mejia",method="t-test_overestim_var",layer="log1p")
    data[name]["experimental"]["weights_Mejia"] = {}
    for pert in data[name]["perts"]:
        pert_weights = data[name]["experimental"]["adata"].uns["DEG_analysis_Mejia"]["scores"][pert]
        pert_weights = np.abs(pert_weights) # Absolute value transformation
        pert_weights = (pert_weights-min(pert_weights))/(max(pert_weights)-min(pert_weights)+1e-8) # Min-max transformation to [0,1]
        pert_weights = pert_weights**2 # Squaring
        pert_weights = pert_weights/sum(pert_weights) # Normalization
        pert_weights = pert_weights[np.argsort(data[name][model]["adata"].uns["DEG_analysis_Mejia"]["names"][pert])] # Bring them in the correct order (so that the corresponding genes are ordered alphabetically)
        data[name]["experimental"]["weights_Mejia"][pert] = pert_weights
    

In [42]:
metrics = {
"MWMSE":lambda cell, exp_µ_p, exp_µ_all, weights: np.dot(weights,(cell-exp_µ_p)**2),
"MWΔR2":lambda cell, exp_µ_p, exp_µ_all, weights: 1-(np.dot(weights,(cell-exp_µ_p)**2))/(np.dot(weights,(exp_µ_p-exp_µ_all-np.dot(weights,(exp_µ_p-exp_µ_all)))**2)) # 0.0 if (avg_expr_pred == mean_unperturbed).all() else 
}
metric_names = list(metrics.keys())
metric_names = [metric_names[0]]

In [45]:
# For every perturbation: calculate the WMSE between the experimental data mean and each cell of the model, then take the mean of that (mean WMSE, aka MWMSE)
cellwise_metric = {}
for name in names:
    cellwise_metric[name] = {}
    exp_pert_mean = np.mean(np.vstack(list(data[name]["experimental"]["pert_means"].values())),axis=0) # µ_all
    for model in models:
        cellwise_metric[name][model] = {}
        for m in metric_names:
            cellwise_metric[name][model][m] = []
            for pert in tqdm(data[name]["perts"]):
                cells = data[name][model]["adata"][data[name][model]["pert_indices"][1][pert]].layers["log1p"].todense()
                m_mean = np.mean([metrics[m](np.asarray(cell).squeeze(),data[name]["experimental"]["pert_means"][pert],exp_pert_mean,data[name]["experimental"]["weights_Mejia"][pert]) for cell in cells])
                cellwise_metric[name][model][m].append(m_mean)
    # Negative baseline: WMSE between each experimental control cell and the experimental data mean
    cellwise_metric[name]["ctrl_baseline"] = {}
    for m in metric_names:
        cellwise_metric[name]["ctrl_baseline"][m] = []
        for pert in tqdm(data[name]["perts"]):
            cells = data[name]["experimental"]["adata"][data[name]["experimental"]["pert_indices"][0]].layers["log1p"].todense()
            m_mean = np.mean([metrics[m](np.asarray(cell).squeeze(),data[name]["experimental"]["pert_means"][pert],exp_pert_mean,data[name]["experimental"]["weights_Mejia"][pert]) for cell in cells])
            cellwise_metric[name]["ctrl_baseline"][m].append(m_mean)

# Hypothesis: all normally distributed, mu_p,exp < mu_p,pgrins << mu_c,exp

# is the order of pval_adj or t score alphabetically? Or are they reordered according to pval?

100%|██████████| 10/10 [00:08<00:00,  1.17it/s]


# Pathways:

## GSEA:

- Inspired from https://www.sc-best-practices.org/conditions/gsea_pathway.html

In [51]:
# Retrieving via python
msigdb = dc.op.resource("MSigDB")

# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))].rename(columns={"genesymbol":"target","geneset":"source"})

In [59]:
gsea_dict = {}
for name in names:
    gsea_dict[name] = {}
    # Filter reactome by genes in each dataset
    gsea_dict[name]["reactome"] = reactome[reactome["target"].isin(data[name]["experimental"]["adata"].var_names)]
    # Get the genesets with no. of genes in [15,500]
    geneset_size = gsea_dict[name]["reactome"].groupby("source").size()
    gsea_dict[name]["genesets"] = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)] # ~600 genesets for both datasets
    for model in models:
        gsea_dict[name][model] = {}
        for pert in tqdm(data[name]["perts"]):
            t_stats = sc.get.rank_genes_groups_df(data[name][model]["adata"], pert, key="DEG_analysis_ctrl").set_index("names").sort_values("scores", key=np.abs, ascending=False)[["scores"]].rename_axis([pert], axis=1)
            scores, pvals = dc.mt.gsea(
                t_stats.T,
                gsea_dict[name]["reactome"][gsea_dict[name]["reactome"]["source"].isin(gsea_dict[name]["genesets"])],
            )
            gsea_dict[name][model][pert] = (
                pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
                .droplevel(level=1, axis=1)
                .sort_values("pval")
            )

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [01:16<00:00,  7.65s/it]


In [ ]:
# get rank of pathways
# plot median distance between ranks for GRiNS and ranks for pert

## Coexpression Graph:
- Inspired from https://link.springer.com/protocol/10.1007/978-1-0716-2067-0_19

- Both GO analysis on modules, and graph comparison with CoDiNA
    - GSEA is done perturbation wise (can pGRiNS capture the specific pathways affected by a perturbation?)
    - GO is done across perturbations (are genes generally coexpressed in such a way that networks/pathways are derivable?   )

In [62]:
for name in names:
    for model in models:
        pert_mean_matrix = np.vstack(list(data[name][model]["pert_means"].values()))
        pd.DataFrame(pert_mean_matrix,columns=data[name][model]["adata"].var_names).to_csv(f"../Data/Experimental/{name}/{name}-{model}_pert_mean.csv",sep=" ",index=False)

# Used as input in coexpr.R